# Stochastic Models and Process Noise

## Unknowns in OD problems
The OD process requires the estimation of many parameters in the system in order to get accurate results. Most of those "unknown" parameters follow known physical models, meaning that we know how we can describe their behaviour (e.g., equation of motion for the orbit or troposphere delay), but there are other that don't fall under this case and that, at first level, are considered to be constant (e.g., ballistic and solar pressure coeffients and measurement biases).

In any case, there are several sources of uncertainty in the estimation process which impact the quality of the solution:
- the physics is either not known or mismodeled
- model inputs (initial conditions) are not known perfectly
- prediction model may not exist (e.g., how biases change over time)
- physical models are simplified (e.g., spherical model for drag)

How do we deal with dynamic uncertainty? There are two available methods:

1) **Least Squares approach**, where an averaged, constant value of the unknowns is found by neglecting most of the underlying physics. This is the simplest approach, but the results are not that accurate, especially for the state covariance estimation which is really poor due to the many simplifying assumptions needed made during the process
2) **Filter approach**. Here stochastic processes are used to soack up the randomness of system's variable which we can't model properly

The structure of sequential estimators naturally accomodates the estimation of time varying quantities whose changes are unpredictable. Note that they are different respect to the Least Square approach, that requires a prediction model for every unknown, assuming that model works along the entire timespan.

Let's image we have a set of experimental data that we'd like to process. If we had a specific model to describe how the data evolve over time, we could fit the data set with the model's output, eventually making predictions on the future values:
$\quad$
<center><img src="Images/model_prediction.png" alt="Drawing" style="width: 1100px;"/><center>    
    






What if we don't have any apriori model to describe the data? We need to apply own our model to characterize the data.
Using the Least Square method we may easily estimate a constant value model for the variable which minimize the errors (residuals) between the model and the observations:

<center><img src="Images/leastSquares1.png" alt="Drawing" style="width: 500px;"/><center>    
    
 

Depending on how much variation is in the data, the results can be pretty bad. Eventually, as next step, we can reduce the fit span where the constant value model is applied. By doing this, we get different constant values for each of the intervals:

<center><img src="Images/leastSquares2.png" alt="Drawing" style="width: 500px;"/><center>    




## Stochastic Processes
In general, a **stochastic process** is a mathematical model used to describe the evolution over time of a system whose state is subject to random variations; they are used to represent behaviours that have at least some component of randomness.

There are many different stochastic processes that we can use in orbit determination (characterization of measurement devices performances, estimation of force model components such ballistic coefficient or solar radiation pressure, transponders/retroreflector delays etc.), but we desire to use processes that have characteristics of the random variables that we seek to model:

- <u>Bounded or unbounded range of values</u>: accordingly with the underlying physics, the random variable can be limited to belong to a specific range or not. For example, a bounded variable is the ballistic coefficient: we know that it can't be negative or extremely large, so it has to stay within a specific range. On the other end, the clock phase biases are unbounded: it always drifts in the same direction walking off from its nominal value forever
- <u>Rate of possible change in value</u>: how much the random variable can vary over time? is it possible that the ballistic coeffient doubles its value in just 5 seconds? Probably not...the system dynamics have a big role here
- <u>Known or unknown mean value</u>: do we have a good estimate of the mean value of the random variable? Again, for the ballistic coefficient we may derive the the average value of the cross-sectional area of the satellite respect to the velocity vector and guess a mean value of it. Or we may have a good model for the atmospheric density and use the data from the model as mean values during the propagation. In those cases we'll estimate the correction to apply to the mean, while in the other case (mean not known) we estimate the whole value

Stochastic processes are applied to many different models which contain uncertainty, such as

 - Atmospheric density
 - Drag effects
 - Solar pressure effects
 - Thrusters
 - Sensor biases



### Gauss-Markov

We use this process when the mean is known and we'd like to estimate the deviation from it. The **state evolution** is defined as:

$$ x\left ( t_{k+1} \right )= \Phi \left ( t_{k+1},t_k \right )x\left ( t_k \right )+ \sqrt{1-\Phi ^2\left ( t_{k+1},t_k \right )}w\left ( t_k \right )$$

, where $\Phi$ is the **state transition matrix** and $w$ is a Gaussian white random variable. This is a type of random signal characterized by two key properties: it follows a Gaussian (normal) distribution and has a constant power spectral density, meaning it contains all frequencies equally. The variance of this random variable is

$$\sigma_{w}^{2}=E\left\{ w^2\left ( t_0 \right )\right\}$$

, being $\sigma_{w}$ the uncertainty around the mean. It is important to note that the $w$ term is only introduced to describe the state evolution, since we don't exactly know the physics of the process. During the filtering process this term is removed, leaving only the exponential decay term.

This process models an exponential decay of the estimate in absence of measurements; the value of the random variable at time $t_{k+1}$ can be written as

$$ \Phi \left ( t_{k+1}, t_k \right )=e^{\alpha \left | t_{k+1}-t_k\right |},\alpha < 0$$

, being $t_{k+1}$ and $t_{k}$ measurement times. Follows that, since $\alpha < 0$:

- When $\left | t_{k+1}-t_k\right |$  is small (small measurement interval), $\Phi$ is close to 1
- When $\left | t_{k+1}-t_k\right |$ become larger (for example in absence of new measurements), $\Phi$ approaches zero with an exponential trend.

Looking at the state evolution equation, is clear to notice how the estimate at $t_{k+1}$ is function of the interval in between measurements. For large intervals the contribution due to the previous estimates decays to zero, and the second term dominates, meaning that the new estimate is mainly based on the value of the random variable, while for short intervals the new estimate is pretty close to the previous one.

The rate of decay is defined by the term $\alpha$. We introduce the **half life** $\tau$ as the time needed to let the estimate decay by one half:

$$e^{\alpha \tau}=0.5\Rightarrow \alpha = ln\frac{0.5}{\tau}$$

The bigger the $\tau$, the longer the previous estimate contribute to the new one.

In the same fashion the variable is propagated, its covariance also evolves over time. In this case we write:

$$P\left ( t_{k+1}\right )=\Phi^{2}\left ( t_{k+1}, t_k \right )P\left ( t_k \right )+\left ( 1- \Phi^{2}\left ( t_{k+1}, t_k \right )\right )\sigma_w^{2}$$

Since $\Phi$ is always less than 1, the first term of the equation reduces the covariance, which is counterintuitive. The second term (which is the **process noise $Q$**) introduces additional uncertainty to the estimate. For large measurement lags, $\Phi$ approaches 0 and the overall covariance approaches $\sigma_w^{2}$.


### Vasicek

We use this process when the mean is not known. In this case we want to estimate it, as well as the deviation from the estimated mean. The state evolution is defined as:

$$V\left ( t_{k+1} \right )=\Phi \left ( t_{k+1},t_k \right )V\left ( t_k \right )+\left ( 1 - \Phi \left ( t_{k+1},t_k \right ) \right )b+\sqrt{\left ( 1-\Phi^2 \left ( t_{k+1},t_k \right ) \right )}w\left ( t_k \right )$$

In this process we have two unknown variables: the total value of the random variable ($V$) and its mean value represented by $b$. When $\Phi$ approaches 0, the total value is the sum of its (estimated) mean plus the random , gaussian variable (noise).
Also in this case the process noise inflates the covariance matrix but, since there are two unknowns variables, we have a matrix form describing it:

$$ Q=\begin{bmatrix}
\left ( 1-\Phi_{k+1,k}^2 \right )\sigma_w^2 & 0 \\
0 & 0 \\
\end{bmatrix}$$

The upper left term is the process noise of the total value; this is the same as the process noise in the Gauss-Markov process. The lower right term is the process noise of the mean value, which is zero, meaning that the (estimated) mean is constant all over the analysis interval. 

### Random Walk

We use this process when the mean is not known and the process is not bounded. The state evolution is defined as:

$$ x\left ( t_{k+1} \right )=x\left ( t_k \right )+\sigma^2 \Delta t Z\left ( t_k \right )$$

$\sigma$ is the **diffusion coefficient**. It is a multiplying factor (as well as $\Delta t$) of the random variable $Z$. Using this process, the total deviation from the initial value is estimated.

#### Best Practices ####

- Use Vasicek if mean is not well known
- Most force model variations have a connection to the orbit period
  - Use one-half orbit period half life in Vasicek
  - Use 2 orbit periods half-life in Gauss-Markov
- Variations in SRP and drag area can be larger than expected
  - Consider the attitude profile

## Process Noise

The process noise is a component of the overall uncertainty model that is meant to accomodate the unknown aspects of the evolution of the estimation states. It usually enters the estimation algorithm as an additive  term $Q$ in the covariance evolution equation:

$$P\left ( t_{k+1}\right )=\Phi\left ( t_{k+1}, t_k \right )P\left ( t_k \right ) \Phi^T\left ( t_{k+1}, t_k \right )+Q \left ( t_{k+1}\right )$$

### Process Noise vs Stochastic Model

In orbit determination the uncertainty in the orbit state is affected by the uncertainty in model parameters and fundamental deficiencies in models:

- Model parameters can be represented by stochastic sequences; the sequence definition includes the process noise evolution for the parameter (e.g., ballistic coefficient)
- Fundamental model deficiencies can sometimes be accomodated with specialized process noise model (e.g., gravity process noise)

## Effects of Stochastic Models on the Parameter Estimation

To demonstrate how different stochastic model can affect the estimation of unknown parameters, we here refer to GPS pseudorange measurements as received by a spacecraft spinning at constant rate. Since it has a cylindrical shape, the area exposed to the velocity vector changes over time and this has an effect on the ballistic coefficient $B_c$, which changes accordingly.

The model characterizing the drag force is a simple sphere: this assumption introduces model parameter errors, so want to see how well each stochastic model fits the evolution of the parameter.

### Gauss-Markov model

This model assumes that the mean value of $B_c$ is known. The vertical axis on the left shows the amount of the applied corrections (in percentage) respect to the defined mean. While new measurements are processed, the corrections oscillate due to the attitude changes; the mean value of those oscillations is higher than the defined one (n. 1). This means that the defined mean value is smaller than the real mean value. At the same time, the uncertainty around the estimate is shown (n. 2).

Once the measurements are not available anymore, the estimated corrections drop to zero in exponential manner (n. 3): this is cleary showing how this is an exponential decay process. The speed of the decay is dictated by the half-life parameter. Also, the uncertainty around the estimate doesn't grow indefinitely; the process bounds it to the input value of the uncertainty.

<center><img src="Images/bc_gaussMarkov.png" alt="Drawing" style="width: 900px;"/><center>   

## Vasicek Model

By using this model, we assume that the mean of $B_c$ is unknown. Now there are two states to estimate: the total value and the variation around the mean. The evolution of the first state (n.1) is the same as in the previous case, but now we have an additional (magenta) line showing the estimate of the (long term) mean. This value increases over time, showing that it is greater than the declared value. In absence of measurements (n.3) the exponential decay goes towards the estimated mean instead of going to zero. The uncertainties around the estimated mean are also available (n.4).

<center><img src="Images/bc_vasicek.png" alt="Drawing" style="width: 900px;"/><center> 

## Random Walk Method

This method is only recommended for clock bias estimation. As shown in the Figure below, the estimate drifts away from the previously calculated values (n.1). Also, not being a bounded process, the covariance grows indefinitely when the are no measurements available (n.2). Last but not least, the estimates doesn't decay as the other two do; instead, it is kept as the last calculated value (n.3) - the State Transition Matrix for this process is the identity matrix).
    

<center><img src="Images/bc_randomWalk.png" alt="Drawing" style="width: 900px;"/><center>     
 
    
